In [5]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

Loading data set

In [7]:
train_df =pd.read_csv('churn-bigml-20.csv')
test_df =pd.read_csv('churn-bigml-80.csv')

In [8]:
X_train = train_df.drop('Churn', axis=1)
y_train = train_df['Churn'].astype(int)

X_test = test_df.drop('Churn', axis=1)
y_test = test_df['Churn'].astype(int)

Treating categorical 

In [9]:
X_train['Area code'] = X_train['Area code'].astype(str)
X_test['Area code'] = X_test['Area code'].astype(str)

Processing Pipeline Definition

In [10]:
categorical_cols = ['State', 'Area code', 'International plan', 'Voice mail plan']
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ]
)

Training and evaluating model

In [11]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

results = []
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    })

baseline_df = pd.DataFrame(results)
print("--- Baseline Models Performance ---")
print(baseline_df.to_string(index=False))

--- Baseline Models Performance ---
              Model  Accuracy  Precision   Recall  F1-Score
Logistic Regression  0.863091   0.582734 0.208763  0.307400
      Decision Tree  0.909602   0.704735 0.652062  0.677376
      Random Forest  0.888972   0.870968 0.278351  0.421875


Hyperparameter Tuning using Grid Search (Random Forest)

In [12]:
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20, None],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2],
    'classifier__class_weight': [None, 'balanced']
}

rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

grid_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)


best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)

print("\n--- Best Hyperparameters ---")
print(grid_search.best_params_)

print("\n--- Tuned Random Forest Classification Report ---")
print(classification_report(y_test, y_pred_tuned, target_names=['Retained', 'Churned']))


--- Best Hyperparameters ---
{'classifier__class_weight': 'balanced', 'classifier__max_depth': 10, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}

--- Tuned Random Forest Classification Report ---
              precision    recall  f1-score   support

    Retained       0.92      0.97      0.95      2278
     Churned       0.74      0.53      0.62       388

    accuracy                           0.90      2666
   macro avg       0.83      0.75      0.78      2666
weighted avg       0.90      0.90      0.90      2666

